# Mirai Variant Validation Walkthrough

This notebook validates that the Mirai static workflow works on different variants by selecting a target sample and running the full pipeline with sample-scoped outputs.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts").exists() and (p / "input").exists():
            ROOT = p
            break

sample_env = os.environ.get("MIRAI_SAMPLE", "").strip()
SAMPLE = (ROOT / "input" / sample_env) if sample_env else sorted((ROOT / "input").glob("*.elf"))[0]
SAMPLE_SHA = SAMPLE.stem
CAPA_JSON = ROOT / "input" / f"capa_{SAMPLE_SHA}.json"
OUTDIR = ROOT / "reports" / f"variant_{SAMPLE_SHA[:12]}"
SAMPLE


In [ ]:
cmd = [
    "python3",
    str(ROOT / "scripts" / "run_full_analysis.py"),
    "--sample", str(SAMPLE),
    "--outdir", str(OUTDIR),
]
if CAPA_JSON.exists():
    cmd.extend(["--capa-json", str(CAPA_JSON)])

subprocess.run(cmd, check=True, cwd=ROOT)


In [ ]:
triage = json.loads((OUTDIR / "json" / "triage_report.json").read_text())
ro = json.loads((OUTDIR / "json" / "rodata_artifacts.json").read_text())
dispatch = json.loads((OUTDIR / "json" / "command_dispatch_map.json").read_text())

summary = {
    "sample_sha256": triage["sample"]["sha256"],
    "file": triage["sample"]["file"],
    "method_tokens": triage["strings_summary"].get("method_tokens", []),
    "command_tokens": triage["strings_summary"].get("command_tokens", []),
    "public_ipv4_candidates": triage["strings_summary"].get("public_ipv4_candidates", []),
    "variant_method_tokens": ro.get("variant_artifacts", {}).get("method_tokens", []),
    "dispatch_rows": len(dispatch.get("commands", [])),
}
summary


In [ ]:
yara_rule = ROOT / "detection" / "mirai_like_d40cf9_rules.yar"
if shutil := __import__("shutil"):
    if shutil.which("yara"):
        out = subprocess.check_output(["yara", "-w", str(yara_rule), str(SAMPLE)], text=True, errors="ignore")
        print(out.strip() if out.strip() else "No YARA matches")
    else:
        print("yara binary not found in PATH")


## Useful IDA Script Order

1. `ida_python/mirai_stage1_annotator.py`
2. `ida_python/mirai_string_hunt_annotator.py`
3. `ida_python/mirai_dns_resolver_pattern_annotator.py`

Use `reports/variant_<hash>/disasm/` and `reports/variant_<hash>/json/` as pivots when validating another variant.